# Etapa 2. Silver – Transformación de datos

**Objetivo:**
- Convertir las columnas de texto a sus tipos de datos correspondientes.
- Limpiar valores vacíos y normalizar registros.
- Validar temperaturas, fechas y coordenadas.
- Separar los registros válidos de los que requieren revisión.
- Crear nuestra primera tabla Delta Silver.


In [0]:
--Validación de tablas disponibles en capa bronze
SHOW TABLES IN  ecomotive.bronze

database,tableName,isTemporary
bronze,iot_sensors_raw,false
,clasificacion_registros,true
,deduplicacion_sensores,true
,formatos_sensores,true
,normalizacion_sensores,true
,registrosvalidos,true


In [0]:
-- validación de la tabla broze  origen

describe table ecomotive.bronze.iot_sensors_raw

col_name,data_type,comment
sensor_id,string,null
truck_id,string,null
timestamp,string,null
temperature_c,string,null
latitude,string,null
longitude,string,null
status,string,null
battery_level_pct,string,null
firmware_version,string,null
signal_strength_dbm,string,null


In [0]:
-- selección de columnas requeridas a través de vista temporal 

CREATE OR REPLACE TEMPORARY VIEW formatos_sensores AS (

    SELECT
        TRIM(sensor_id) as sensor_id,
        TRIM(truck_id) as vehiculo_id,
        try_cast(NULLIF(TRIM(timestamp), '') as timestamp)    AS momento_de_lectura,
        try_cast (nullif(TRIM(temperature_c),'') as double) as temperatura_c,
        try_cast(TRIM(latitude) as double) as latitud,
        try_cast(TRIM(longitude) as double) as longitud, 
        TRIM(UPPER(status)) as estado,
        try_cast(TRIM (battery_level_pct) as INT) as porcentaje_bateria_sensor,
        NULLIF(TRIM (firmware_version),'') as  firmware_version,
        TRY_CAST(NULLIF(TRIM(signal_strength_dbm), '') AS DOUBLE) AS signal_strength_dbm
    FROM ecomotive.bronze.iot_sensors_raw
) ;

select * from formatos_sensores


sensor_id,vehiculo_id,momento_de_lectura,temperatura_c,latitud,longitud,estado,porcentaje_bateria_sensor,firmware_version,signal_strength_dbm
S-1001,T-101,2025-01-12T08:00:00.000Z,2.5,40.4168,-3.7038,OK,null,null,null
S-1001,T-101,2025-01-12T08:15:00.000Z,2.8,40.42,-3.71,OK,null,null,null
S-1001,T-101,2025-01-12T08:30:00.000Z,3.1,40.43,-3.72,OK,null,null,null
S-1002,T-102,2025-01-12T09:00:00.000Z,3.9,41.3851,2.1734,WARNING,null,null,null
S-1002,T-102,2025-01-12T09:15:00.000Z,4.2,41.39,2.18,ALARM,null,null,null
S-1002,T-102,2025-01-12T09:30:00.000Z,4.5,41.395,2.185,ALARM,null,null,null
S-1003,T-103,2025-01-12T08:10:00.000Z,-1.0,39.4699,-0.3763,OK,null,null,null
S-1004,T-104,2025-01-12T10:00:00.000Z,2.0,37.3891,-5.9845,OK,null,null,null
S-1005,T-105,2025-01-12T11:00:00.000Z,2.2,43.263,-2.935,OK,null,null,null
S-1001,T-101,2025-01-12T08:45:00.000Z,3.0,40.44,-3.73,OK,null,null,null


In [0]:
--Normalización de typeo

CREATE OR REPLACE TEMPORARY VIEW normalizacion_sensores AS

SELECT
    * EXCEPT (estado),
    CASE
        WHEN estado IN ('OK', 'ACTIVE', 'ACTIVO') THEN 'ACTIVO'
        WHEN estado IN ('WARNING', 'ALARM') THEN 'FALLANDO'
        WHEN estado IN ('UNKNOWN_STATUS', '') THEN 'DESCONOCIDO'
        ELSE 'REVISAR'
    END AS estado

FROM formatos_sensores;

SELECT *
FROM normalizacion_sensores 

sensor_id,vehiculo_id,momento_de_lectura,temperatura_c,latitud,longitud,porcentaje_bateria_sensor,firmware_version,signal_strength_dbm,estado
S-1001,T-101,2025-01-12T08:00:00.000Z,2.5,40.4168,-3.7038,null,null,null,ACTIVO
S-1001,T-101,2025-01-12T08:15:00.000Z,2.8,40.42,-3.71,null,null,null,ACTIVO
S-1001,T-101,2025-01-12T08:30:00.000Z,3.1,40.43,-3.72,null,null,null,ACTIVO
S-1002,T-102,2025-01-12T09:00:00.000Z,3.9,41.3851,2.1734,null,null,null,FALLANDO
S-1002,T-102,2025-01-12T09:15:00.000Z,4.2,41.39,2.18,null,null,null,FALLANDO
S-1002,T-102,2025-01-12T09:30:00.000Z,4.5,41.395,2.185,null,null,null,FALLANDO
S-1003,T-103,2025-01-12T08:10:00.000Z,-1.0,39.4699,-0.3763,null,null,null,ACTIVO
S-1004,T-104,2025-01-12T10:00:00.000Z,2.0,37.3891,-5.9845,null,null,null,ACTIVO
S-1005,T-105,2025-01-12T11:00:00.000Z,2.2,43.263,-2.935,null,null,null,ACTIVO
S-1001,T-101,2025-01-12T08:45:00.000Z,3.0,40.44,-3.73,null,null,null,ACTIVO


In [0]:
CREATE OR REPLACE TEMPORARY VIEW deduplicacion_sensores AS (

    SELECT * FROM (

        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY
                    sensor_id,
                    vehiculo_id,
                    momento_de_lectura
                ORDER BY
                    temperatura_c DESC,
                    porcentaje_bateria_sensor DESC
            ) AS rn

        FROM normalizacion_sensores

    )

    WHERE rn = 1

);

SELECT COUNT(*) FROM deduplicacion_sensores;

COUNT(*)
68


# Reglas de negocio

- sensor_id	No puede ser NULL ni vacío
- temperatura_c	Debe estar entre -40 y 85 °C
- latitud	Debe estar entre -90 y 90
- longitud	Debe estar entre -180 y 180
- porcentaje_bateria_sensor	Debe estar entre 0 y 100
- estado	Debe pertenecer al catálogo permitido

In [0]:

--validación de cantidad de registros que no cumplen reglas de negocio - rechazados 

SELECT
    COUNT(*) AS total_registros,

    SUM(CASE
        WHEN sensor_id IS NULL OR sensor_id = ''
        THEN 1 ELSE 0
    END) AS sensores_sin_id,

    SUM(CASE
        WHEN temperatura_c IS NULL
        THEN 1 ELSE 0
    END) AS temperaturas_nulas,

    SUM(CASE
        WHEN temperatura_c < -40 OR temperatura_c > 85
        THEN 1 ELSE 0
    END) AS temperaturas_fuera_rango,

    SUM(CASE
        WHEN latitud < -90 OR latitud > 90
        THEN 1 ELSE 0
    END) AS latitudes_invalidas,

    SUM(CASE
        WHEN longitud < -180 OR longitud > 180
        THEN 1 ELSE 0
    END) AS longitudes_invalidas,

    SUM(CASE
        WHEN porcentaje_bateria_sensor < 0
          OR porcentaje_bateria_sensor > 100
        THEN 1 ELSE 0
    END) AS baterias_invalidas

FROM deduplicacion_sensores;

total_registros,sensores_sin_id,temperaturas_nulas,temperaturas_fuera_rango,latitudes_invalidas,longitudes_invalidas,baterias_invalidas
68,7,1,0,0,0,0


In [0]:

-- identificación de registros  rechazados
SELECT *
FROM deduplicacion_sensores 
WHERE
    sensor_id IS NULL
    OR sensor_id = ''
    OR temperatura_c IS NULL
    OR temperatura_c < -40
    OR temperatura_c > 85
    OR latitud < -90
    OR latitud > 90
    OR longitud < -180
    OR longitud > 180
    OR porcentaje_bateria_sensor < 0
    OR porcentaje_bateria_sensor > 100;

sensor_id,vehiculo_id,momento_de_lectura,temperatura_c,latitud,longitud,porcentaje_bateria_sensor,firmware_version,signal_strength_dbm,estado,rn
null,T-102,2026-11-08T10:35:00.000Z,24.0,40.4168,-3.7038,92,1.0.2,-57.0,ACTIVO,1
null,T-107,2025-01-12T12:15:00.000Z,2.1,41.3851,2.1734,null,null,null,ACTIVO,1
null,TRK-601,2026-11-07T11:00:00.000Z,21.0,4.6,-74.1,null,null,null,ACTIVO,1
null,TRK-602,2026-11-07T11:05:00.000Z,22.0,4.609999999999999,-74.11,null,null,null,ACTIVO,1
null,TRK-603,2026-11-07T11:10:00.000Z,23.0,4.619999999999999,-74.11999999999999,null,null,null,ACTIVO,1
null,TRK-604,2026-11-07T11:15:00.000Z,24.0,4.63,-74.13,null,null,null,ACTIVO,1
null,TRK-605,2026-11-07T11:20:00.000Z,25.0,4.64,-74.14,null,null,null,ACTIVO,1
S-1002,T-102,2026-11-08T10:30:00.000Z,null,40.4168,-3.7038,95,1.0.2,-55.0,ACTIVO,1


In [0]:
--- Clasificación de registros- Valido o  rechazado

CREATE OR REPLACE TEMPORARY VIEW clasificacion_registros AS

SELECT *,
    CASE
        WHEN sensor_id IS NULL 
            OR sensor_id = ''
            OR vehiculo_id IS NULL
            OR vehiculo_id = ''
            OR temperatura_c IS NULL
            OR temperatura_c < -40 OR temperatura_c > 85
            OR latitud IS NULL
            OR latitud < -90 OR latitud > 90
            OR longitud IS NULL
            OR longitud < -180 OR longitud > 180
            OR porcentaje_bateria_sensor < 0
            OR porcentaje_bateria_sensor > 100
            OR momento_de_lectura IS NULL
            
        THEN 'RECHAZADO'
        ELSE 'VALIDO'
    END AS estado_calidad

FROM deduplicacion_sensores;

In [0]:
select
    estado_calidad, 
    count(*)
from clasificacion_registros
group BY 1

estado_calidad,count(*)
RECHAZADO,16
VALIDO,52


# Persistir los  registros en tablas delta
- [una](url) para rechazados
- otra para validos

In [0]:
DROP TABLE ecomotive.silver.validos_iot_sensors

In [0]:
CREATE OR REPLACE TABLE ecomotive.silver.rechazos_iot_sensor
USING DELTA
AS
    SELECT * EXCEPT (estado_calidad)
    FROM clasificacion_registros
    WHERE estado_calidad = 'RECHAZADO';

select * from ecomotive.silver.rechazos_iot_sensor

sensor_id,vehiculo_id,momento_de_lectura,temperatura_c,latitud,longitud,porcentaje_bateria_sensor,firmware_version,signal_strength_dbm,estado,rn
null,T-102,2026-11-08T10:35:00.000Z,24.0,40.4168,-3.7038,92,1.0.2,-57.0,ACTIVO,1
null,T-107,2025-01-12T12:15:00.000Z,2.1,41.3851,2.1734,null,null,null,ACTIVO,1
null,TRK-601,2026-11-07T11:00:00.000Z,21.0,4.6,-74.1,null,null,null,ACTIVO,1
null,TRK-602,2026-11-07T11:05:00.000Z,22.0,4.609999999999999,-74.11,null,null,null,ACTIVO,1
null,TRK-603,2026-11-07T11:10:00.000Z,23.0,4.619999999999999,-74.11999999999999,null,null,null,ACTIVO,1
null,TRK-604,2026-11-07T11:15:00.000Z,24.0,4.63,-74.13,null,null,null,ACTIVO,1
null,TRK-605,2026-11-07T11:20:00.000Z,25.0,4.64,-74.14,null,null,null,ACTIVO,1
S-1002,,2025-01-12T09:45:00.000Z,4.0,41.4,2.19,null,null,null,FALLANDO,1
S-1002,T-102,2026-11-08T10:30:00.000Z,null,40.4168,-3.7038,95,1.0.2,-55.0,ACTIVO,1
S-1003,null,2026-11-08T10:40:00.000Z,24.2,40.4168,-3.7038,90,1.0.2,-58.0,ACTIVO,1


In [0]:
CREATE OR REPLACE  TEMPORARY  VIEW registrosvalidos as (
SELECT * EXCEPT (estado_calidad, rn),
 MD5(
            CONCAT_WS(
                '|',
                sensor_id,
                vehiculo_id,
                DATE_FORMAT(
                    momento_de_lectura,
                    'yyyy-MM-dd HH:mm:ss.SSSSSS'
                )
            )
        ) AS row_hash

FROM clasificacion_registros
WHERE estado_calidad = 'VALIDO');

select * from registrosvalidos


sensor_id,vehiculo_id,momento_de_lectura,temperatura_c,latitud,longitud,porcentaje_bateria_sensor,firmware_version,signal_strength_dbm,estado,row_hash
S-1001,T-101,2025-01-12T08:00:00.000Z,2.5,40.4168,-3.7038,98,1.0.2,-55.0,ACTIVO,07063cbd767033504e173fe61f87fc5d
S-1001,T-101,2025-01-12T08:15:00.000Z,2.8,40.42,-3.71,97,1.0.2,-57.0,ACTIVO,592fe2f2ebc0d43479b63fe2c1d3583e
S-1001,T-101,2025-01-12T08:30:00.000Z,3.1,40.43,-3.72,96,1.0.2,-59.0,ACTIVO,fac280cfd4bcd47bd5c46456a50f631c
S-1001,T-101,2025-01-12T08:45:00.000Z,3.0,40.44,-3.73,null,null,null,ACTIVO,a4bf094ab983636314cea1706a02b5b1
S-1001,T-101,2026-11-08T10:00:00.000Z,24.5,40.4168,-3.7038,96,1.0.2,-55.0,ACTIVO,a00bf1bd878922e7c0622efd101c0cf2
S-1002,T-102,2025-01-12T09:00:00.000Z,3.9,41.3851,2.1734,85,1.1.0,-68.0,FALLANDO,ceb0862e34aa9d282a719c87133e3e77
S-1002,T-102,2025-01-12T09:15:00.000Z,4.2,41.39,2.18,82,1.1.0,-70.0,FALLANDO,e505aafcd4e9d7991d72a03b4e2bd825
S-1002,T-102,2025-01-12T09:30:00.000Z,4.5,41.395,2.185,80,1.1.0,-72.0,FALLANDO,12d182f1f687a68423256f273e9e759e
S-1002,T-102,2025-01-12T09:45:00.000Z,4.0,41.4,2.19,null,null,null,FALLANDO,329747f6390b200485b72e1d98ab8b55
S-1003,T-103,2025-01-12T08:10:00.000Z,-1.0,39.4699,-0.3763,91,1.0.5,-61.0,ACTIVO,0f2afc61444f6e13354b4beb1b15e4c7


In [0]:


----Creación de la estructura de la tabla final de silver 


CREATE TABLE IF NOT EXISTS ecomotive.silver.validos_iot_sensors (

    id_record BIGINT GENERATED ALWAYS AS IDENTITY (start with 1 increment by 1)  COMMENT "sk del registro",
    sensor_id STRING,
    vehiculo_id STRING,
    momento_de_lectura TIMESTAMP,
    temperatura_c DOUBLE,
    latitud DOUBLE,
    longitud DOUBLE,
    porcentaje_bateria_sensor INT,
    firmware_version STRING,
    signal_strength_dbm DOUBLE,
    estado STRING,
    _record_added_at timestamp default CURRENT_TIMESTAMP() 

)

USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');





In [0]:
---Celda adicional para  agregar columana que usaré para el hasch

ALTER TABLE ecomotive.silver.validos_iot_sensors
ADD COLUMNS (
    row_hash STRING
);


In [0]:
MERGE INTO ecomotive.silver.validos_iot_sensors as target
USING registrosvalidos AS source

ON  target.sensor_id = source.sensor_id 
AND target.momento_de_lectura = source.momento_de_lectura

WHEN NOT MATCHED THEN

INSERT (row_hash,
        sensor_id,
        vehiculo_id,
        momento_de_lectura,
        temperatura_c,
        latitud,
        longitud,
        porcentaje_bateria_sensor,
        firmware_version,
        signal_strength_dbm,
        estado
        )
VALUES (
        source.row_hash,
        source.sensor_id,
        source.vehiculo_id,
        source.momento_de_lectura,
        source.temperatura_c,
        source.latitud,
        source.longitud,
        source.porcentaje_bateria_sensor,
        source.firmware_version,
        source.signal_strength_dbm,
        source.estado
)




num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
52,0,0,52


In [0]:
SELECT
count (*)
FROM ecomotive.silver.validos_iot_sensors 

count(*)
52


In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_record) AS sk_unicas,
    COUNT(DISTINCT sensor_id, momento_de_lectura) AS eventos_unicos
FROM ecomotive.silver.validos_iot_sensors;

total_registros,sk_unicas,eventos_unicos
52,52,52
